# 🧠 Module 01: The Async Mental Model & Concurrency

Welcome to **Async Python from Scratch**!

Before writing production code in **FastAPI**, **Django**, or **FDE Data Pipelines**, you need a rock-solid mental model of what the Python Event Loop actually does.

---

## 1. The Core Problem: Waiting on I/O

In backend engineering, services spend 90%+ of their time **waiting**:
- Waiting for a PostgreSQL query (10–50ms)
- Waiting for Redis (1–2ms)
- Waiting for an external API (Stripe, OpenAI, AWS S3) (150–1500ms)

### 🍽️ The Restaurant Analogy:
- **Synchronous (Blocking)**: 1 Waiter takes an order at Table 1, walks to the kitchen, and **stands staring at the chef for 15 minutes** until the food is ready. Tables 2–10 starve.
- **Asynchronous (Cooperative Multitasking)**: 1 Waiter takes Table 1's order, hands the ticket to the chef, and **immediately serves Tables 2, 3, and 4**. When the chef rings the bell (*"Order 1 is ready!"*), the waiter pauses for a fraction of a second, serves Table 1, and continues.

> 💡 **Key Takeaway**: One single thread can handle thousands of concurrent requests if it never sits idle while waiting for I/O!

---

### ⚡ Note on Jupyter Notebooks vs Python Scripts:
> In standard Python scripts (`.py`), you must start the event loop with `asyncio.run(main())`.
> **In Jupyter Notebooks, an event loop is ALREADY running in the background!**
> This means you can use top-level `await` directly inside notebook cells (e.g. `await my_function()`).

--- 
## 2. Synchronous (Blocking) Demo

Here we simulate fetching 3 independent services: User Profile (2.0s), Payment Status (1.5s), and Notifications (1.0s).
In synchronous code, execution pauses at each step. Total time is the **sum** of all waits (~4.5s).

In [1]:
import time

def fetch_user_profile(user_id: int) -> dict:
    print(f"[{time.strftime('%X')}] 🟡 Fetching profile for user {user_id}...")
    time.sleep(2.0)  # Simulates 2.0s network/database latency
    print(f"[{time.strftime('%X')}] 🟢 Profile retrieved for user {user_id}.")
    return {"user_id": user_id, "name": f"User_{user_id}"}

def fetch_payment_status(user_id: int) -> dict:
    print(f"[{time.strftime('%X')}] 🟡 Fetching payments for user {user_id}...")
    time.sleep(1.5)  # Simulates 1.5s payment gateway latency
    print(f"[{time.strftime('%X')}] 🟢 Payments retrieved for user {user_id}.")
    return {"user_id": user_id, "status": "active"}

def fetch_notifications(user_id: int) -> list:
    print(f"[{time.strftime('%X')}] 🟡 Fetching notifications for user {user_id}...")
    time.sleep(1.0)  # Simulates 1.0s push service latency
    print(f"[{time.strftime('%X')}] 🟢 Notifications retrieved for user {user_id}.")
    return [{"id": 1, "text": "Welcome back!"}]

print("=" * 60)
print("🚀 STARTING SYNCHRONOUS (BLOCKING) EXECUTION")
print("=" * 60)
start_time = time.perf_counter()

profile = fetch_user_profile(42)
payments = fetch_payment_status(42)
notifications = fetch_notifications(42)

total_time = time.perf_counter() - start_time
print("=" * 60)
print(f"🏁 Finished in {total_time:.2f} seconds!")
print(f"Results: {profile['name']}, Status: {payments['status']}, Unread: {len(notifications)}")
print("Notice: 2.0s + 1.5s + 1.0s = ~4.5s total time.")
print("=" * 60)

🚀 STARTING SYNCHRONOUS (BLOCKING) EXECUTION
[07:50:29] 🟡 Fetching profile for user 42...
[07:50:31] 🟢 Profile retrieved for user 42.
[07:50:31] 🟡 Fetching payments for user 42...
[07:50:33] 🟢 Payments retrieved for user 42.
[07:50:33] 🟡 Fetching notifications for user 42...
[07:50:34] 🟢 Notifications retrieved for user 42.
🏁 Finished in 4.52 seconds!
Results: User_42, Status: active, Unread: 1
Notice: 2.0s + 1.5s + 1.0s = ~4.5s total time.


--- 
## 3. Asynchronous (Concurrent) Demo

Now we convert the functions to coroutines with `async def` and non-blocking `await asyncio.sleep(...)`.
With `asyncio.gather(...)`, all 3 requests start together on a **single thread**.

👉 Total time drops from **~4.5s** to **~2.0s** (only as long as the slowest individual task)!

In [2]:
import asyncio
import time

async def fetch_user_profile_async(user_id: int) -> dict:
    print(f"[{time.strftime('%X')}] 🟡 [Async] Fetching profile for user {user_id}...")
    await asyncio.sleep(2.0)  # Non-blocking pause: yields execution back to the loop!
    print(f"[{time.strftime('%X')}] 🟢 [Async] Profile retrieved for user {user_id}.")
    return {"user_id": user_id, "name": f"User_{user_id}"}

async def fetch_payment_status_async(user_id: int) -> dict:
    print(f"[{time.strftime('%X')}] 🟡 [Async] Fetching payments for user {user_id}...")
    await asyncio.sleep(1.5)
    print(f"[{time.strftime('%X')}] 🟢 [Async] Payments retrieved for user {user_id}.")
    return {"user_id": user_id, "status": "active"}

async def fetch_notifications_async(user_id: int) -> list:
    print(f"[{time.strftime('%X')}] 🟡 [Async] Fetching notifications for user {user_id}...")
    await asyncio.sleep(1.0)
    print(f"[{time.strftime('%X')}] 🟢 [Async] Notifications retrieved for user {user_id}.")
    return [{"id": 1, "text": "Welcome back!"}]

print("=" * 60)
print("⚡ STARTING ASYNCHRONOUS (CONCURRENT) EXECUTION")
print("=" * 60)
start_time = time.perf_counter()

# Notice in Jupyter we can 'await' directly at top level!
profile, payments, notifications = await asyncio.gather(
    fetch_user_profile_async(42),
    fetch_payment_status_async(42),
    fetch_notifications_async(42),
)

total_time = time.perf_counter() - start_time
print("=" * 60)
print(f"🏁 Finished in {total_time:.2f} seconds!")
print(f"Results: {profile['name']}, Status: {payments['status']}, Unread: {len(notifications)}")
print(f"Notice: Instead of ~4.5s, it completed in ~{total_time:.1f}s (the longest individual wait)!")
print("=" * 60)

⚡ STARTING ASYNCHRONOUS (CONCURRENT) EXECUTION
[07:51:37] 🟡 [Async] Fetching profile for user 42...
[07:51:37] 🟡 [Async] Fetching payments for user 42...
[07:51:37] 🟡 [Async] Fetching notifications for user 42...
[07:51:38] 🟢 [Async] Notifications retrieved for user 42.
[07:51:39] 🟢 [Async] Payments retrieved for user 42.
[07:51:39] 🟢 [Async] Profile retrieved for user 42.
🏁 Finished in 2.00 seconds!
Results: User_42, Status: active, Unread: 1
Notice: Instead of ~4.5s, it completed in ~2.0s (the longest individual wait)!


--- 
## 4. The Critical Pitfall: CPU-Bound vs I/O-Bound

A very common misconception: *"If I add `async def` to my slow machine-learning calculation or big loop, it will run faster!"*

🚨 **FALSE!**
- `asyncio` runs on a **SINGLE thread**.
- If a function is calculating numbers, it never calls `await` to yield control.
- The Event Loop freezes, blocking all other requests.

Let's see this in action below:

In [3]:
def heavy_math(task_id: int, n: int = 8_000_000) -> int:
    print(f"[{time.strftime('%X')}] ⚙️ Task {task_id} starting heavy CPU calculation...")
    res = sum(i * i for i in range(n))
    print(f"[{time.strftime('%X')}] ✅ Task {task_id} finished CPU calculation.")
    return res

async def fake_async_cpu(task_id: int):
    # Bad practice: CPU-bound code without an await
    return heavy_math(task_id)

print("=" * 60)
print("🧪 TEST: 3 CPU-Heavy Tasks Wrapped in 'async def'")
print("=" * 60)
start = time.perf_counter()

# Even with gather, they run SEQUENTIALLY because no await yields control!
await asyncio.gather(
    fake_async_cpu(1),
    fake_async_cpu(2),
    fake_async_cpu(3)
)
print(f"⏱️ CPU-bound total time: {time.perf_counter() - start:.2f}s (Ran strictly one after another!)")
print("👉 Key Rule: For CPU-bound work, use Multiprocessing or ThreadPoolExecutor, NOT raw async def!")

🧪 TEST: 3 CPU-Heavy Tasks Wrapped in 'async def'
[07:52:44] ⚙️ Task 1 starting heavy CPU calculation...
[07:52:45] ✅ Task 1 finished CPU calculation.
[07:52:45] ⚙️ Task 2 starting heavy CPU calculation...
[07:52:45] ✅ Task 2 finished CPU calculation.
[07:52:45] ⚙️ Task 3 starting heavy CPU calculation...
[07:52:46] ✅ Task 3 finished CPU calculation.
⏱️ CPU-bound total time: 1.58s (Ran strictly one after another!)
👉 Key Rule: For CPU-bound work, use Multiprocessing or ThreadPoolExecutor, NOT raw async def!


--- 
## 5. 🎯 Hands-On Challenge: E-Commerce Order Aggregator

**Scenario (Forward Deployed Engineering / API Gateway)**:
When a user clicks "Confirm Order", we must query 4 microservices:
1. `verify_address`: takes `0.8s`
2. `check_inventory`: takes `1.2s`
3. `preauth_credit_card`: takes `1.5s`
4. `check_fraud_risk`: takes `0.5s`

In synchronous code, this takes `0.8 + 1.2 + 1.5 + 0.5 = 4.0 seconds`.
Your mission: convert them to async and run them concurrently using `asyncio.gather` so the checkout finishes in under **1.6 seconds**!

In [4]:
# Solution & Interactive Code:
async def verify_address_async(user_id: int) -> dict:
    await asyncio.sleep(0.8)
    return {"status": "valid", "zip": "94103"}

async def check_inventory_async(item_ids: list[str]) -> dict:
    await asyncio.sleep(1.2)
    return {"available": True, "count": len(item_ids)}

async def preauth_credit_card_async(user_id: int, amount: float) -> dict:
    await asyncio.sleep(1.5)
    return {"authorized": True, "tx_id": "tx_99812"}

async def check_fraud_risk_async(user_id: int) -> dict:
    await asyncio.sleep(0.5)
    return {"fraud_score": 12, "decision": "ALLOW"}

# Execution:
start = time.perf_counter()
print("🛒 Starting order validation (Async)...")

addr, inv, card, fraud = await asyncio.gather(
    verify_address_async(101),
    check_inventory_async(["SKU-1", "SKU-2"]),
    preauth_credit_card_async(101, 149.99),
    check_fraud_risk_async(101),
)

duration = time.perf_counter() - start
print(f"⚡ Async order validation finished in {duration:.2f}s!")
print(f"Summary: Card={card['authorized']}, Fraud={fraud['decision']}, Inventory={inv['available']}, Address={addr['status']}")

🛒 Starting order validation (Async)...
⚡ Async order validation finished in 1.50s!
Summary: Card=True, Fraud=ALLOW, Inventory=True, Address=valid


--- 
## 6. 🤔 What If Task B Depends on Task A's Output?

A common real-world question:
> *"What if `preauth_credit_card` needs the exact `total_price` returned by `check_inventory`? How can we pass it if both run in `gather()`?"*

### ❌ The Impossible Attempt:
You cannot write:  
`asyncio.gather(check_inventory(), preauth_credit_card(inv['total']))`  
Because `inv` does not exist yet at the moment `gather()` is called!

---

### ✅ Pattern A: Staged Concurrency (Two-Stage Gather)
If multiple tasks produce inputs needed for later tasks, split your flow into **Stages**:
1. **Stage 1**: Run independent tasks in parallel (`inventory`, `address`, `fraud`).
2. **Stage 2**: Once Stage 1 finishes, pass the outputs into dependent tasks (`credit_card`).

```
Stage 1 (Parallel): [Inventory: 1.2s] [Address: 0.8s] [Fraud: 0.5s]  -> Takes 1.2s
Stage 2 (Dependent): [Credit Card: 1.5s (uses inv['total'])]          -> Takes 1.5s
Total Time: 1.2s + 1.5s = 2.7s (vs 4.0s sync)
```

---

### 🏆 Pattern B: Coroutine Composition (Pipeline Branching - Recommended!)
Why should `inventory` and `card` wait for `address` or `fraud`?
We can compose the dependent sequence into its own coroutine, then `gather` that branch with the other independent branches!

In [5]:
# 1. Task A: Inventory calculates total price
async def check_inventory_with_price(items: list[str]) -> dict:
    print(f"[{time.strftime('%X')}] 🟡 Starting inventory check...")
    await asyncio.sleep(1.2)
    total_price = 149.99
    print(f"[{time.strftime('%X')}] 🟢 Inventory check completed. Total calculated: ${total_price}")
    return {"available": True, "count": len(items), "total": total_price}

# 2. Task B: Depends directly on total_price from Task A
async def charge_card_with_amount(user_id: int, amount: float) -> dict:
    print(f"[{time.strftime('%X')}] 💳 Charging credit card with ${amount} from inventory...")
    await asyncio.sleep(1.5)
    print(f"[{time.strftime('%X')}] 🟢 Credit card charged successfully!")
    return {"authorized": True, "amount_charged": amount}

# 3. Compose them together into a single pipeline branch!
async def inventory_and_payment_pipeline(user_id: int, items: list[str]):
    inv = await check_inventory_with_price(items)         # Step 1: wait for inventory
    card = await charge_card_with_amount(user_id, inv["total"]) # Step 2: pass inventory output to card
    return inv, card

print("=" * 60)
print("📦 DEMO: Dependent Tasks with Coroutine Composition")
print("=" * 60)
start = time.perf_counter()

# Branch 1 (inv -> card) runs concurrently alongside Branch 2 (address) and Branch 3 (fraud)!
(inv, card), addr, fraud = await asyncio.gather(
    inventory_and_payment_pipeline(101, ["SKU-1", "SKU-2"]),
    verify_address_async(101),
    check_fraud_risk_async(101),
)

print("=" * 60)
print(f"🏁 Finished in {time.perf_counter() - start:.2f} seconds!")
print(f"Inventory: {inv['count']} items, Card Charged: ${card['amount_charged']}, Address: {addr['status']}, Fraud: {fraud['decision']}")
print("=" * 60)

📦 DEMO: Dependent Tasks with Coroutine Composition
[07:59:27] 🟡 Starting inventory check...
[07:59:29] 🟢 Inventory check completed. Total calculated: $149.99
[07:59:29] 💳 Charging credit card with $149.99 from inventory...
[07:59:30] 🟢 Credit card charged successfully!
🏁 Finished in 2.70 seconds!
Inventory: 2 items, Card Charged: $149.99, Address: valid, Fraud: ALLOW


--- 
## 🏆 Summary Checklist for Module 01

- [x] Understood **Cooperative Multitasking** vs Threading vs Multiprocessing.
- [x] Understood why **I/O-bound** benefits from async, while **CPU-bound** blocks the loop.
- [x] Mastered `async def`, `await`, and `asyncio.gather()`.
- [x] Handled **Data Dependencies** using Staged Concurrency & Coroutine Composition.
- [x] Understood how top-level `await` works naturally in Jupyter Notebooks.

👉 **Next Up: Module 02 — Coroutines, Tasks (`create_task`), and Event Loop Internals!**